# Questão 3 - Load Schemas with Datasets

In [2]:
#Premissas obrigatórias

#* Realize o carregamento de todos os CSVs.
#* Utilize obrigatoriamente Python 3.
#* Utilize qualquer biblioteca necessária (nativa ou externa) para conexão e carregamento dos dados.
#* Não faça tratamentos como: Remoção de nulos ou correção de caracteres especiais

In [3]:
#Escreva um script python para realizar o carregamento de todos os arquivos CSV respeitando o schema criado na questão anterior. 

In [4]:
#Importando bibliotecas necessárias
import csv
import os
import psycopg2
import json

In [5]:
#Definindo o diretório onde os arquivos CSVs estão localizados
DATASET_DIR = "Dataset"

#Banco de dados associado ao PostgreSQL (Configurado para conexão com localhost)
with open("Config/db_config.json", "r", encoding="utf-8") as file:
    config = json.load(file)

#Arquivo de configuração do banco de dados PostgreSQL não será carregado no GitHub por questões de segurança
#ip route | grep default > para verificar qual é o IP do host
DB_CONFIG = {
    "host": config["host"],
    "port": config["port"],
    "database": config["database"],
    "user": config["user"],
    "password": config["password"]
}

In [6]:
#Funções para tratar nomes de tabelas e colunas para o PostgreSQL sendo as mesmas usadas em Schemas.ipynb
def created_table_name(filename):
    """ Utiliza o nome do arquivo CSV como nome da tabela SQL."""
    table_name = os.path.splitext(filename)[0]

    return table_name.lower()

def created_column_name(column):
    """ Normaliza o nome das colunas para PostgreSQL."""

    #Tratamento de possiveis espaços e caracteres especiais no nome da coluna
    column = column.strip()
    column = column.replace(" ", "_")
    column = column.replace("-", "_")

    return column.lower()

In [7]:
#Função para inserir os dados dos arquivos CSVs no PostgreSQL
def load_csv_to_postgres(connection, filepath):
    """ Insere os dados de um arquivo CSV em sua respectiva tabela PostgreSQL presente no arquivo schemas.sql. """

    filename = os.path.basename(filepath)
    table_name = created_table_name(filename)

    print(f"Carregando: {filename}")

    #Cursor para consultar os tipos das colunas
    cur = connection.cursor()

    with open(
        filepath,
        "r",
        encoding="utf-8-sig",
        newline=""
    ) as csvfile:
        reader = csv.reader(csvfile)
        header = next(reader)
        columns = [
            created_column_name(column)
            for column in header
        ]
        column_list = ", ".join(
            f'"{column}"'
            for column in columns
        )
        placeholders = ", ".join(
            ["%s"] * len(columns)
        )

        #Consulta os tipos das colunas da tabela no PostgreSQL para tratar valores nulos (SQL)
        cur.execute("""
            SELECT
                column_name,
                data_type
            FROM information_schema.columns
            WHERE table_name = %s
        """, (table_name,))

        column_types = {
            column_name: data_type
            for column_name, data_type in cur.fetchall()
        }

        #Ação de INSERT no PostgreSQL (SQL)
        insert_query = f"""
            INSERT INTO "{table_name}"
            ({column_list})
            VALUES ({placeholders})
        """

        #INSERT SQL e tratamento de valores nulos
        for row in reader:
            new_row = []
            for column, value in zip(columns, row):
                column_type = column_types.get(column)

                #Se o valor estiver vazio e a coluna for
                #DATE, TIMESTAMP ou INTEGER são os mais afetados, então o valor serão tratados como NULL (SQL)
                if value == "" and column_type in (
                    "date",
                    "timestamp without time zone",
                    "timestamp with time zone",
                    "integer"
                ):
                    value = None
                new_row.append(value)
            cur.execute(
                insert_query,
                new_row
            )

        connection.commit()
        cur.close()
    print(f"  {filename} inserido com sucesso!")

In [8]:
#Função para chamar todos os arquivos CSVs e a partir da função load_csv_to_postgres inserir no PostgreSQL
def load_all_csvs():
    """ Carrega todos os arquivos CSV do diretório DATASET_DIR para o banco de dados PostgreSQL. """

    connection = psycopg2.connect(**DB_CONFIG)

    #Busca apenas arquivos CSV no diretório DATASET_DIR e chama a função load_csv_to_postgres para cada arquivo encontrado
    try:
        files = sorted(os.listdir(DATASET_DIR))
        for filename in files:
            if not filename.lower().endswith(".csv"):
                continue
            filepath = os.path.join(
                DATASET_DIR,
                filename
            )
            load_csv_to_postgres(
                connection,
                filepath
            )
    except Exception as error:
        connection.rollback()
        print(
            f"Erro durante o carregamento: {error}")
        raise

    finally:
        connection.close()

In [9]:
#Chamando função principal para gerar o INSERT SQL a partir dos arquivos CSVs
load_all_csvs()

Carregando: addresses.csv
  addresses.csv inserido com sucesso!
Carregando: attributes.csv
  attributes.csv inserido com sucesso!
Carregando: brands.csv
  brands.csv inserido com sucesso!
Carregando: categories.csv
  categories.csv inserido com sucesso!
Carregando: customers.csv
  customers.csv inserido com sucesso!
Carregando: employees.csv
  employees.csv inserido com sucesso!
Carregando: fiscal_invoices.csv
  fiscal_invoices.csv inserido com sucesso!
Carregando: goods_receipt_items.csv
  goods_receipt_items.csv inserido com sucesso!
Carregando: goods_receipts.csv
  goods_receipts.csv inserido com sucesso!
Carregando: locations.csv
  locations.csv inserido com sucesso!
Carregando: order_items.csv
  order_items.csv inserido com sucesso!
Carregando: orders.csv
  orders.csv inserido com sucesso!
Carregando: payments.csv
  payments.csv inserido com sucesso!
Carregando: product_suppliers.csv
  product_suppliers.csv inserido com sucesso!
Carregando: product_variants.csv
  product_variants.

In [10]:
#Qual o total de linhas somadas das seguintes tabelas: customers, orders, order_items e payments?
import pandas as pd
customers = pd.read_csv('Dataset/customers.csv')
orders = pd.read_csv('Dataset/orders.csv')
order_items = pd.read_csv('Dataset/order_items.csv')
payments = pd.read_csv('Dataset/payments.csv')
total_linhas = len(customers) + len(orders) + len(order_items) + len(payments)
print(f"Total de linhas somadas das tabelas: {total_linhas}")

#SELECT
  #  (SELECT COUNT(*) FROM customers)
  #+ (SELECT COUNT(*) FROM orders)
  #+ (SELECT COUNT(*) FROM order_items)
  #+ (SELECT COUNT(*) FROM payments) AS total_registros;

Total de linhas somadas das tabelas: 251864
